# ⚡ Análise Exploratória — Consumo de Energia Elétrica no Brasil (2004–2023)

## Contexto

Este notebook analisa a série histórica de consumo de energia elétrica no Brasil, com foco em entender como o país consome energia ao longo do tempo, quais setores lideram o crescimento e como diferentes regiões apresentam padrões distintos.

Os dados são provenientes do **Ministério de Minas e Energia (MME)** via Base dos Dados, cobrindo o período de **janeiro de 2004 a dezembro de 2023** com granularidade mensal por estado e segmento de consumo.

---

## Estrutura da Análise

A análise foi estruturada em torno da **decomposição de séries temporais**, separando cada componente para entender o fenômeno de forma isolada:

- **Tendência** — como o consumo evoluiu no longo prazo, por segmento e por região
- **Sazonalidade** — quais meses são sistematicamente mais altos ou baixos, e como esse padrão varia pelo Brasil
- **Resíduo** — o que não é explicado pela tendência nem pela sazonalidade, revelando eventos extraordinários

---

## Perguntas que a análise responde

- O consumo de energia cresceu de forma uniforme entre os segmentos e regiões?
- Qual segmento é mais sensível a crises econômicas?
- O Brasil tem um único padrão sazonal ou padrões distintos por região?
- Quais eventos históricos deixaram marca visível no consumo de energia?
- Quais regiões cresceram mais desde 2004?


## 1. Configurações

*Esta seção realiza todas as importações, define constantes e carrega os dados. Não há análise aqui — apenas a infraestrutura necessária para as seções seguintes.*

## 1.1 Bibliotecas


In [ ]:

# ── IMPORTS ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
 


## 1.2 Constantes


In [ ]:
# ── CONSTANTES ──────────────────────────────────────────────────────
MESES_NOME = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
 
SEGMENTOS  = ['Residencial', 'Industrial', 'Comercial']
CORES_SEG  = ['steelblue', 'firebrick', 'seagreen']
CORES_REG  = px.colors.qualitative.D3
 
REGIOES = {
    'Norte':        ['AM', 'RR', 'AP', 'PA', 'TO', 'RO', 'AC'],
    'Nordeste':     ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA'],
    'Centro-Oeste': ['MT', 'MS', 'GO', 'DF'],
    'Sudeste':      ['SP', 'RJ', 'ES', 'MG'],
    'Sul':          ['PR', 'SC', 'RS'],
}
 
ORDEM_ESTADOS = [
    'RS', 'SC', 'PR',
    'SP', 'RJ', 'MG', 'ES',
    'MT', 'MS', 'GO', 'DF',
    'BA', 'PE', 'CE', 'MA', 'PB', 'RN', 'AL', 'SE', 'PI',
    'PA', 'AM', 'RO', 'TO', 'AC', 'AP', 'RR',
]
 
 


## 1.3  Criando funções auxiliares


In [ ]:
# ── FUNÇÕES AUXILIARES ───────────────────────────────────────────────
def decompor(serie):
    """Retorna decomposição multiplicativa mensal."""
    return seasonal_decompose(serie.asfreq('MS'), model='multiplicative', period=12)
 
 
def serie_segmento(df, segmento):
    """Agrega consumo nacional por segmento."""
    return df[df['tipo_consumo'] == segmento].groupby('data')['consumo'].sum()
 
 
def serie_regiao(df, regiao):
    """Agrega consumo total por região."""
    return (df[(df['regiao'] == regiao) & (df['tipo_consumo'] == 'Total')]
            .groupby('data')['consumo'].sum())
 
 
def heatmap_sazonal(df, tipo_consumo, agrupador, ordem=None, titulo=None, height=400):
    """Gera heatmap de índice sazonal."""
    d = df[df['tipo_consumo'] == tipo_consumo].copy()
    d['mes'] = d['data'].dt.month
 
    saz = d.groupby([agrupador, 'mes'])['consumo'].mean().reset_index()
    base = saz.groupby(agrupador)['consumo'].mean().rename('baseline')
    saz  = saz.merge(base, on=agrupador)
    saz['indice'] = saz['consumo'] / saz['baseline']
 
    hm = saz.pivot(index=agrupador, columns='mes', values='indice')[list(range(1, 13))]
    hm = hm.loc[[x for x in ordem if x in hm.index]] if ordem else hm.loc[base.sort_values(ascending=False).index]
 
    fig = px.imshow(hm,
                    labels=dict(x='Mês', y=agrupador, color='Índice sazonal'),
                    x=MESES_NOME,
                    color_continuous_scale='RdYlBu_r',
                    aspect='auto')
    fig.update_traces(hovertemplate='<b>%{y}</b><br>Mês: %{x}<br>Índice: %{z:.3f}<extra></extra>')
    fig.update_layout(title=titulo or f'Sazonalidade — {tipo_consumo}',
                      template='plotly_white', height=height)
    return fig
 
 
def subplots_componente(df, componente, titulo, ylabel, hline=None):
    """Gera subplots do componente (trend/seasonal/resid) para os 3 segmentos."""
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        subplot_titles=SEGMENTOS, vertical_spacing=0.08)
    for i, (seg, cor) in enumerate(zip(SEGMENTOS, CORES_SEG)):
        res = decompor(serie_segmento(df, seg))
        serie = getattr(res, componente)
        if componente == 'seasonal':
            serie = serie.iloc[:36]
        fig.add_trace(go.Scatter(x=serie.index, y=serie.values,
                                  name=seg, line=dict(color=cor, width=2)),
                      row=i+1, col=1)
    if hline is not None:
        for r in range(1, 4):
            fig.add_hline(y=hline, line_dash='dash', line_color='black',
                          line_width=0.8, row=r, col=1)
    fig.update_layout(title=titulo, template='plotly_white',
                      width=1000, height=700, hovermode='x unified', showlegend=False)
    fig.update_yaxes(title_text=ylabel)
    return fig
 
 


## 1.4  Carregando os dados


In [ ]:
consumo = pd.read_csv('data/consumo_energia.csv')
consumo['data'] = pd.to_datetime(
    consumo[['ano', 'mes']].rename(columns={'ano': 'year', 'mes': 'month'}).assign(day=1))
consumo['regiao'] = consumo['sigla_uf'].map(
    {uf: reg for reg, ufs in REGIOES.items() for uf in ufs})


consumo.head()
 
 


## 2. Visão Geral — Consumo Total


In [ ]:


consumo_total = (consumo[consumo['tipo_consumo'] == 'Total']
                 .groupby('data')['consumo'].sum().reset_index())

fig = px.line(consumo_total, x='data', y='consumo',
              title='<b>Consumo Total de Energia no Brasil (2004–2023)</b>',
              labels={'consumo': 'Consumo (MWh)', 'data': 'Data'},
              template='plotly_white')

# Cores padronizadas
VERDE_OK = "#d4edda"
VERMELHO_ALERTA = "#f8d7da"

blocos = [
    dict(x0="2004-01-01", x1="2008-08-31", label="Crescimento Inicial", color=VERDE_OK),
    dict(x0="2008-09-01", x1="2009-06-30", label="Crise 2008", color=VERMELHO_ALERTA),
    dict(x0="2009-07-01", x1="2013-10-31", label="Recuperação", color=VERDE_OK),
    dict(x0="2013-11-01", x1="2016-12-31", label="Grande Recessão", color=VERMELHO_ALERTA),
    dict(x0="2017-01-01", x1="2020-02-29", label="Estabilização", color=VERDE_OK),
    dict(x0="2020-03-01", x1="2020-12-31", label="Pandemia", color=VERMELHO_ALERTA),
    dict(x0="2021-01-01", x1="2023-12-31", label="Retomada Atual", color=VERDE_OK)
]

fig.update_traces(line=dict(color='black', width=2))

for b in blocos:
    fig.add_vrect(
        x0=b['x0'], x1=b['x1'],
        fillcolor=b['color'], 
        opacity=0.7, 
        layer="below", 
        line_width=0,
        annotation_text=f" {b['label']}",
        annotation_position="bottom left",
        annotation_font=dict(size=10, color="#444"),
        annotation_textangle=-90
    )

fig.update_layout(
    height=600, 
    margin=dict(t=100, b=80),
    yaxis_title="Consumo (MWh)"
)

fig.write_image('midia/consumo_total_contexto.png',
    scale=3)
fig.show()
 


![](midia/consumo_total_contexto.png)

### Observações

O consumo total de energia no Brasil cresceu aproximadamente **70% entre 2004 e 2023**, saindo de ~27 GWh para ~47 GWh mensais. O crescimento não foi linear — três interrupções marcam a série:

- **Crise 2008** — queda abrupta e rápida, com recuperação completa em menos de 2 anos. O Brasil absorveu o choque externo sem sequelas duradouras.
- **Grande Recessão (2014–2017)** — o período mais longo de estagnação da série. Combinação de crise econômica doméstica, crise hídrica e retração industrial. O consumo levou quase 4 anos para retomar a trajetória de crescimento.
- **Pandemia (2020)** — queda mais abrupta registrada, porém a mais curta. A recuperação foi em V — o consumo voltou ao nível pré-pandemia em menos de 1 ano.

A sazonalidade anual é visível ao longo de toda a série nas oscilações regulares da linha — e sua amplitude cresce proporcionalmente ao nível de consumo, confirmando o comportamento multiplicativo da série.

O crescimento total, no entanto, esconde dinâmicas muito distintas entre os segmentos. A seção seguinte investiga como cada um se comportou ao longo do período.

## 3. Comportamento por Segmento


In [ ]:

df_mix = (consumo[~consumo['tipo_consumo'].isin(['Total', 'Cativo'])]
          .groupby(['data', 'tipo_consumo'])['consumo'].sum().reset_index())
 
df_pct = (df_mix.pivot(index='data', columns='tipo_consumo', values='consumo')
          .fillna(0).pipe(lambda d: d.divide(d.sum(axis=1), axis=0) * 100)
          .reset_index().melt(id_vars='data', var_name='tipo_consumo', value_name='percentual'))
 
fig = px.area(df_pct, x='data', y='percentual', color='tipo_consumo',
              title='Participação no Consumo de Energia por Segmento (%)',
              labels={'percentual': '% de Contribuição', 'data': 'Data', 'tipo_consumo': 'Segmento'},
              color_discrete_map={'Residencial': '#1f77b4', 'Industrial': '#d62728',
                                  'Comercial': '#2ca02c', 'Outros': '#7f7f7f'},
              category_orders={'tipo_consumo': ['Industrial', 'Comercial', 'Residencial', 'Outros']},
              template='plotly_white')
fig.update_layout(yaxis_range=[0, 100], hovermode='x unified')
fig.write_image('midia/mix_segmentos.png',
    scale=3)
fig.show()
 
 


<p align="center">
  <img src="midia/mix_segmentos.png" width="1200" height="600">
</p>

### Observações

A participação industrial no consumo total caiu de ~47% em 2004 para ~35% em 2023 — uma redução de 12 pontos percentuais em duas décadas. No mesmo período, residencial e comercial ganharam espaço.

Esse movimento reflete duas forças simultâneas: a desindustrialização relativa da economia brasileira e o crescimento da classe média, que expandiu o consumo doméstico e de serviços.

O efeito é mais visível durante a Recessão (2014–2017) ,onde a faixa industrial encolhe visivelmente enquanto as demais se mantêm, confirmando que a crise afetou desproporcionalmente o setor produtivo.




A análise por segmento mostrou o que mudou. Agora é hora de desmontar o sinal — separar tendência, sazonalidade e resíduo para entender como cada força opera de forma isolada.

## 4. Decomposição da Série Temporal: Anatomia do Consumo


In [ ]:

consumo_brasil = serie_segmento(consumo, 'Total')
res_total      = decompor(consumo_brasil)
 
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Tendência', 'Sazonalidade', 'Resíduo'),
                    vertical_spacing=0.08)
for row, (comp, cor, nome) in enumerate([
        ('trend',    'steelblue', 'Tendência'),
        ('seasonal', 'seagreen',  'Sazonalidade'),
        ('resid',    'firebrick', 'Resíduo')], start=1):
    s = getattr(res_total, comp)
    fig.add_trace(go.Scatter(x=s.index, y=s, name=nome,
                              line=dict(color=cor, width=2)), row=row, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='black', line_width=1, row=3, col=1)
fig.update_layout(title='Decomposição Multiplicativa — Consumo Total de Energia no Brasil',
                  height=700, showlegend=False, template='plotly_white')
fig.write_image('midia/decomposicao_total.png')
fig.show()



![](midia/decomposicao_total.png)

### Observações

*   **Tendência:** Crescimento estrutural até 2014, seguido de um platô (recessão) até 2017. A retomada forte ocorre a partir de 2021, atingindo novas máximas históricas.
*   **Sazonalidade:** Padrão cíclico rígido. Picos em **março** (calor/pico industrial) e vales em **junho** (clima ameno/baixa atividade).
*   **Resíduos (Eventos Atípicos):**
    *   **2008:** Queda brusca reflexo da Crise Financeira Global.
    *   **Fev/2014:** Pico positivo anômalo causado por uma onda de calor histórica.
    *   **2020:** O maior desvio negativo da série, provocado pelo impacto imediato da Pandemia.

*Consolidada a visão geral da decomposição, as próximas seções detalham cada componente individualmente, iniciando por uma investigação multifacetada da **Tendência** de consumo.*

## 5. Analise da tendencia


## 5.1 Tendencia por segmento


In [ ]:
# ════════════════════════════════════════════════════════════════════
# 4. TENDÊNCIA POR SEGMENTO - PADRONIZADO E LIMPO
# ════════════════════════════════════════════════════════════════════

fig = subplots_componente(consumo, 'trend', 
                          '<b>Análise Histórica: Ciclos de Consumo de Energia no Brasil</b>', 
                          'MWh')

# Cores padronizadas (Hex específicos para ficarem bonitos e nítidos)
VERDE_OK = "#d4edda" # Sucesso/Crescimento
VERMELHO_ALERTA = "#f8d7da" # Crise/Queda

blocos = [
    dict(x0="2004-01-01", x1="2008-08-31", label="Crescimento Inicial", color=VERDE_OK),
    dict(x0="2008-09-01", x1="2009-06-30", label="Crise 2008", color=VERMELHO_ALERTA),
    dict(x0="2009-07-01", x1="2013-10-31", label="Recuperação", color=VERDE_OK),
    dict(x0="2013-11-01", x1="2016-12-31", label="Grande Recessão", color=VERMELHO_ALERTA),
    dict(x0="2017-01-01", x1="2020-02-29", label="Estabilização", color=VERDE_OK),
    dict(x0="2020-03-01", x1="2020-12-31", label="Pandemia", color=VERMELHO_ALERTA),
    dict(x0="2021-01-01", x1="2023-12-31", label="Retomada Atual", color=VERDE_OK)
]

fig.update_traces(line=dict(color='black', width=2.5))

for b in blocos:
    fig.add_vrect(
        x0=b['x0'], x1=b['x1'],
        fillcolor=b['color'], 
        opacity=0.7, 
        layer="below", 
        line_width=0,
        annotation_text=f" {b['label']}",
        annotation_position="bottom left", # Texto começa de baixo para não embolar
        annotation_font=dict(size=10, color="#444"),
        annotation_textangle=-90
    )



fig.update_layout(height=800, margin=dict(t=100, b=50))
fig.write_image('midia/tendencia_total.png')

fig.show()

![](midia/tendencia_segmentos.png)

###  Observações

*   **Residencial:** Crescimento constante e resiliente. Ignorou as crises de 2008 e 2014, impulsionado pela universalização do acesso e maior eletrificação dos lares.
*   **Industrial:** Setor mais volátil. Sofreu quedas severas em 2008 e na "Grande Recessão" (2014-2016), com um **pequeno recuo em 2012** (reflexo das incertezas da MP 579). Mostra forte retomada pós-2021.
*   **Comercial:** Trajetória de subida interrompida pela crise de 2014 e pela correção tarifária subsequente. Apresenta o recuo mais nítido durante a pandemia, seguido de recuperação em "V".

## 5.2 Tendencia por região


In [ ]:

fig = go.Figure()

for i, regiao in enumerate(REGIOES):
    # Pegando a tendência absoluta
    trend = decompor(serie_regiao(consumo, regiao)).trend
    
    fig.add_trace(go.Scatter(
        x=trend.index, y=trend.values, 
        name=regiao,
        line=dict(color=CORES_REG[i], width=3),
        hovertemplate=f'<b>{regiao}</b><br>Consumo: %{{y:.2f}} MWh<extra></extra>'))

VERDE_OK = "#d4edda"
VERMELHO_ALERTA = "#f8d7da"

blocos = [
    dict(x0="2004-01-01", x1="2008-08-31", label="Crescimento Inicial", color=VERDE_OK),
    dict(x0="2008-09-01", x1="2009-06-30", label="Crise 2008", color=VERMELHO_ALERTA),
    dict(x0="2009-07-01", x1="2013-10-31", label="Recuperação", color=VERDE_OK),
    dict(x0="2013-11-01", x1="2016-12-31", label="Grande Recessão", color=VERMELHO_ALERTA),
    dict(x0="2017-01-01", x1="2020-02-29", label="Estabilização", color=VERDE_OK),
    dict(x0="2020-03-01", x1="2020-12-31", label="Pandemia", color=VERMELHO_ALERTA),
    dict(x0="2021-01-01", x1="2023-12-31", label="Retomada Atual", color=VERDE_OK)
]

for b in blocos:
    fig.add_vrect(
        x0=b['x0'], x1=b['x1'],
        fillcolor=b['color'], 
        opacity=0.4, 
        layer="below", 
        line_width=0,
        annotation_text=f" {b['label']}",
        annotation_position="bottom left",
        annotation_font=dict(size=9, color="#666"),
        annotation_textangle=-90
    )

fig.update_layout(
    title='<b>Consumo Absoluto da Tendência por Região</b>',
    template='plotly_white', 
    xaxis_title='Ano',
    yaxis_title='Consumo (MWh)', 
    hovermode='x unified',
    legend_title='Região', 
    width=1000, height=600
)

fig.show()

![](midia/crescimento_regiao.png)

### **Observações**

*   **Domínio do Sudeste:** O gráfico evidencia o abismo de consumo absoluto. O Sudeste consome, sozinho, mais do que todas as outras regiões somadas, sendo o principal driver da curva nacional.
*   **Sensibilidade Regional:** O Sudeste e o Sul mostram quedas mais acentuadas nos blocos de crise (2008 e 2014), refletindo a alta concentração industrial dessas regiões.
*   **Resiliência do Norte e Centro-Oeste:** Diferente das outras, as regiões Norte e Centro-Oeste mantêm uma inclinação de crescimento quase linear, sendo menos afetadas visualmente pelas janelas de recessão econômica no valor absoluto.
*   **Convergência Sul e Nordeste:** Nota-se que o Sul e o Nordeste disputam o segundo lugar em volume, com o Sul mantendo uma leve vantagem histórica, mas com ambas apresentando trajetórias de crescimento muito similares.

Até aqui, vimos que o **Sudeste** domina o volume total de energia consumida no país. No entanto, a grande disparidade de escala absoluta pode mascarar o dinamismo das outras regiões. 

Para entender quem realmente está liderando a expansão energética e o desenvolvimento regional nas últimas décadas, precisamos eliminar a barreira do tamanho. Na próxima seção, utilizaremos a **Análise de Base 100** para comparar o **crescimento relativo** de cada região, revelando quais fronteiras estão avançando com maior velocidade.

## 5.3 Tendencia por região (Crescimento relativo)


In [ ]:

fig = go.Figure()
for i, regiao in enumerate(REGIOES):
    trend = decompor(serie_regiao(consumo, regiao)).trend
    norm  = trend / trend.dropna().iloc[0]
    fig.add_trace(go.Scatter(
        x=norm.index, y=norm.values, name=regiao,
        line=dict(color=CORES_REG[i], width=3),
        hovertemplate=f'<b>{regiao}</b><br>Fator: %{{y:.2f}}x<extra></extra>'))
 
fig.add_hline(y=1, line_dash='dot', line_color='black', opacity=0.5)
fig.update_layout(
    title='<b>Crescimento Relativo da Tendência por Região (Base 1)</b>',
    template='plotly_white', xaxis_title='Ano',
    yaxis_title='Fator de Crescimento', hovermode='x unified',
    legend_title='Região', width=1000, height=600)
fig.show()
 


![](midia/crescimento_regiao_relativo.png)

### **Observações: A História por Trás do Crescimento Relativo**

**Liderança do Centro-Oeste:** É o grande destaque da série. Apresenta um crescimento quase linear e extremamente vigoroso, dobrando seu consumo (fator 2.2) no período. Isso reflete a forte expansão da fronteira agrícola e das agroindústrias na região.

**O "Susto" do Norte em 2018:** A região Norte vinha em uma arrancada impressionante, mas sofre uma **queda significativa em 2018**. Mesmo com esse desvio, a recuperação foi rápida, consolidando-se como a segunda região que mais cresceu proporcionalmente.

**Nordeste e Sul (Ritmo Moderado):** Ambas apresentam trajetórias muito próximas, com um crescimento acumulado em torno de 70-75%. O Sul mostra-se ligeiramente mais volátil às crises, enquanto o Nordeste mantém uma ascensão mais constante nos últimos anos.

**O Gigante Estacionado (Sudeste):** Embora seja o maior consumidor em volume, proporcionalmente foi o que **menos cresceu** (apenas ~40%). Isso faz sentido: por ser uma região já altamente industrializada e urbanizada, o mercado é maduro e o crescimento tende a ser marginal e mais sensível a crises econômicas.



O Norte se destaca como a região de maior crescimento relativo — mas a queda brusca entre 2018 e 2020 chama atenção. Para entender o que aconteceu, é preciso descer um nível: analisar os estados individualmente, onde o choque está concentrado.

In [ ]:
fig = go.Figure()
for i, uf in enumerate(REGIOES['Norte']):
    serie = (consumo[(consumo['sigla_uf'] == uf) & (consumo['tipo_consumo'] == 'Total')]
             .set_index('data')['consumo']
             .sort_index()
             .loc['2016':'2020'])
    trend = decompor(serie).trend
    norm  = trend / trend.dropna().iloc[0]
    fig.add_trace(go.Scatter(
        x=norm.index, y=norm.values, name=uf,
        line=dict(color=CORES_REG[i], width=2.5),
        hovertemplate=f'<b>{uf}</b><br>Fator: %{{y:.2f}}x<extra></extra>'))

fig.add_hline(y=1, line_dash='dot', line_color='black', opacity=0.5)
fig.update_layout(
    title='<b>Crescimento Relativo da Tendência — Estados do Norte (2016–2020)</b>',
    template='plotly_white', xaxis_title='Ano',
    yaxis_title='Fator de Crescimento (base jan/2016)', hovermode='x unified',
    legend_title='Estado', width=1000, height=600)
fig.show()

### Observações

A queda do Norte entre 2016 e 2020 tem um culpado claro: o Pará.

Enquanto os demais estados mantinham trajetória de crescimento, o Pará recuou
para 0,85 em 2018 — uma retração de 15% — e só começou a se recuperar em 2019.
A causa foi um único evento: o embargo ambiental da Hydro Alunorte.

Em fevereiro de 2018, chuvas extremas em Barcarena geraram suspeitas de vazamento
de rejeitos de bauxita nos rios da região. O MPF embargou a refinaria em 50% —
e como a Alunorte opera de forma integrada com a mina de Paragominas e a fábrica
de alumínio Albras, as três plantas reduziram produção simultaneamente. O complexo
inteiro consome centenas de megawatts, e o impacto aparece com precisão na tendência.

O embargo durou mais de um ano. A produção de alumina caiu de 6,4 para 3,25
milhões de toneladas — queda de 50% — e só foi liberada em abril de 2019.

Analisadas as tendências — e o que elas escondem — o próximo passo é separar
outro componente da série: a sazonalidade. Se a tendência responde *quanto* o
consumo cresceu, a sazonalidade responde *quando* — quais meses são
sistematicamente mais altos ou baixos, e se esse ritmo varia pelo Brasil.

# 6. Análise de sazonalidade


# 6.1 Sazonalidade por segmento


In [ ]:

fig = subplots_componente(consumo, 'seasonal',
                          '<b>Sazonalidade do Consumo de Energia por Segmento</b>',
                          'Fator sazonal', hline=1)
fig.write_html('sazonalidade_segmentos.html')
fig.show()


![](midia/sazonal_segmento.png)

### Observações

Os três segmentos respondem a lógicas diferentes.

O **residencial** tem o padrão mais intenso: pico em janeiro (férias + calor)
e vale em junho. O **comercial** segue ritmo parecido, mas com amplitude menor.
O **industrial** destoa dos dois: janeiro cai com as férias coletivas, e o pico
se desloca para agosto-outubro, quando a atividade produtiva está em plena carga.

Quanto mais contínuo o processo, menos a sazonalidade consegue penetrar — daí
a amplitude industrial ser a metade da residencial.


# 6.2 Sazonalidade por região


In [ ]:
# ════════════════════════════════════════════════════════════════════
# 9. HEATMAPS DE SAZONALIDADE
# ════════════════════════════════════════════════════════════════════
 
# 9a. Total por Região
fig = heatmap_sazonal(consumo, 'Total', 'regiao', height=350,
                       titulo='<b>Sazonalidade do Consumo Total por Região</b>')
fig.write_html('heatmap_total_regiao.html')
fig.show()
 


![](midia/heatmap_regiao.png)

### Observações

A sazonalidade "brasileira" não existe — ela é a média de dois comportamentos opostos.

Sul e Sudeste concentram o maior volume de consumo do país, então ditam o padrão
nacional: pico no verão (jan-mar) e vale no inverno. Norte e Nordeste seguem o
caminho inverso, com pico em set-out, quando a estação seca atinge o máximo de
calor. O Centro-Oeste destoa dos dois: o agronegócio empurra o pico pro segundo
semestre, enquanto o inverno seco do cerrado não derruba o consumo como no Sul.

5 regiões, 5 sazonalidades , agregá-las numa só curva esconde mais do que revela.

# 6.3 Sazonalidade por estado


In [ ]:
 
fig = heatmap_sazonal(consumo, 'Total', 'sigla_uf',
                       ordem=ORDEM_ESTADOS, height=900,
                       titulo='<b>Sazonalidade do Consumo de Energia por Estado</b><br>'
                              '<sup>Sul → Sudeste → Centro-Oeste → Nordeste → Norte</sup>')
fig.write_html('heatmap_total_estado.html')
fig.show()

![](midia/heatmap_estado.png)

### Observações

O heatmap por estado revela o que a análise regional media: dentro de cada
região, há comportamentos bem distintos.

O **RJ** tem a sazonalidade mais intensa do Sudeste ,destaque-se o pico de verão bem acima
de SP e MG, reflexo do turismo e do calor urbano amplificando o consumo residencial.
O **RS** lidera o Sul, com picos intensos no verão com o vale de inverno mais prolongado do país.

O **MT** carrega a assinatura do agronegócio: azul em jan-fev (entressafra) e
vermelho intenso em set-out (safra). O **MS** chama atenção pelo vale anômalo
em jun-jul, mais profundo que qualquer outro estado do Centro-Oeste, coincidindo com o período do inverno seco na região

O **PI** tem o pico de novembro mais intenso do Nordeste, bem acima dos vizinhos
, combinação de calor extremo no fim da estação seca com crescimento recente
da atividade econômica local.

# 6.4 Sazonalidade por segmento e região


In [ ]:
# 9b. Por Segmento e Região
for seg in SEGMENTOS:
    fig = heatmap_sazonal(consumo, seg, 'regiao', height=350,
                           titulo=f'<b>Sazonalidade — {seg}</b>')
    fig.write_html(f'heatmap_{seg.lower()}_regiao.html')
    fig.show()


![](midia/heatmap_segmento.png)

### Observações

Comparando os três segmentos, o padrão de inversão Norte/Sul se repete em todos
— mas com intensidades diferentes.

O **residencial** tem a maior amplitude: o Sul chega a 1.1 em janeiro enquanto
o Norte despenca para 0.9 em fevereiro.

 O **comercial** segue lógica parecida,
mas com o Centro-Oeste apresentando o vale mais profundo em jun-jul, shoppings
e escritórios vazios no inverno seco do cerrado.

 O **industrial** é o mais
homogêneo entre regiões, mas o Centro-Oeste se destaca com o pico mais intenso
em ago-set: assinatura clara do agronegócio.

Compreendida a sazonalidade em todas as suas camadas, resta o terceiro componente
da decomposição: o resíduo. É nele que aparece tudo aquilo que a tendencia e e sazonalidade não conseguem explicar,incluindo eventos atípicos

# 7 Análise de resíduos


# 7.1 Resíduos por segmento


In [ ]:
N_TOP = 5

fig = make_subplots(
    rows=1, cols=len(SEGMENTOS),
    shared_yaxes=True,
    subplot_titles=SEGMENTOS,
    horizontal_spacing=0.04)

for i, seg in enumerate(SEGMENTOS):
    resid = decompor(serie_segmento(consumo, seg)).resid.dropna()
    cor = CORES_SEG[i]

    # linha cinza de fundo
    fig.add_trace(go.Scatter(
        x=resid.index, y=resid.values,
        line=dict(color='rgba(180,180,180,0.35)', width=1),
        showlegend=False,
        hovertemplate=f'<b>{seg}</b><br>%{{x|%b %Y}}: %{{y:.4f}}<extra></extra>'),
        row=1, col=i+1)

    idx_picos = resid.nlargest(N_TOP).index
    idx_vales = resid.nsmallest(N_TOP).index

    for idx in list(idx_picos) + list(idx_vales):
        fig.add_trace(go.Scatter(
            x=[idx], y=[resid[idx]],
            mode='markers',
            marker=dict(color=cor, size=8, symbol='circle',
                       line=dict(color='white', width=1)),
            showlegend=False,
            hovertemplate=f'<b>{seg}</b><br>%{{x|%b %Y}}: %{{y:.4f}}<extra></extra>'),
            row=1, col=i+1)

    # anotação só no maior pico e maior vale
    for idx, ay in [(resid.idxmax(), -30), (resid.idxmin(), 30)]:
        fig.add_annotation(
            x=idx, y=resid[idx],
            text=idx.strftime('%b/%Y'),
            showarrow=True, arrowhead=0,
            arrowcolor=cor, arrowwidth=1.2,
            ax=0, ay=ay,
            font=dict(size=9, color=cor),
            xref=f'x{i+1}', yref=f'y{i+1}')

    fig.add_hline(y=1, line_dash='dash', line_color='black',
                  line_width=1, opacity=0.3, row=1, col=i+1)

fig.update_layout(
    title=dict(text='<b>Resíduo do Consumo de Energia por Segmento — Brasil</b>',
               font=dict(size=15)),
    template='plotly_white',
    height=430, width=900,
    showlegend=False,
    margin=dict(t=80, b=40))

fig.update_yaxes(title_text='Fator residual', row=1, col=1,
                 gridcolor='rgba(200,200,200,0.3)')
fig.update_xaxes(gridcolor='rgba(200,200,200,0.3)')

fig.show()


![](midia/residuo_segmentos.png)

### Observações

O residual por segmento confirma que cada setor responde de forma distinta
aos mesmos eventos.

O pico de **fev/2014** aparece no residencial e no comercial simultaneamente,
confirmando a onda de calor como evento de abrangência ampla.

O **residencial** registra seu maior pico em jan/2015, outra onda de calor intensa.

O **industrial e comercial** têm os maiores vales em 2020 pelo COVID. Os picos
que aparecem logo após provavelmente são artefatos: o modelo se acostumou com
o nível deprimido do lockdown e interpretou a recuperação normal como anomalia
positiva.

O **comercial** tem o vale mais extremo em mai/2020, superando até o industrial,
reflexo do fechamento imediato de shoppings e serviços no lockdown.

# 7.2 Resíduos por região


In [ ]:
N_TOP = 3

regioes_list = list(REGIOES.keys())
fig = make_subplots(
    rows=1, cols=len(regioes_list),
    shared_yaxes=True,
    subplot_titles=regioes_list,
    horizontal_spacing=0.04)

for i, regiao in enumerate(regioes_list):
    resid = decompor(serie_regiao(consumo, regiao)).resid.dropna()
    cor = CORES_REG[i]

    # linha de fundo cinza
    fig.add_trace(go.Scatter(
        x=resid.index, y=resid.values,
        line=dict(color='rgba(180,180,180,0.35)', width=1),
        showlegend=False,
        hovertemplate=f'<b>{regiao}</b><br>%{{x|%b %Y}}: %{{y:.4f}}<extra></extra>'),
        row=1, col=i+1)

    idx_picos = resid.nlargest(N_TOP).index
    idx_vales = resid.nsmallest(N_TOP).index

    # todos os pontos na cor da região
    for idx in list(idx_picos) + list(idx_vales):
        fig.add_trace(go.Scatter(
            x=[idx], y=[resid[idx]],
            mode='markers',
            marker=dict(color=cor, size=8, symbol='circle',
                       line=dict(color='white', width=1)),
            showlegend=False,
            hovertemplate=f'<b>{regiao}</b><br>%{{x|%b %Y}}: %{{y:.4f}}<extra></extra>'),
            row=1, col=i+1)

    # anotação só no maior pico e maior vale
    for idx, ay in [(resid.idxmax(), -30), (resid.idxmin(), 30)]:
        fig.add_annotation(
            x=idx, y=resid[idx],
            text=idx.strftime('%b/%Y'),
            showarrow=True, arrowhead=0,
            arrowcolor=cor, arrowwidth=1.2,
            ax=0, ay=ay,
            font=dict(size=9, color=cor),
            xref=f'x{i+1}', yref=f'y{i+1}')

    fig.add_hline(y=1, line_dash='dash', line_color='black',
                  line_width=1, opacity=0.3, row=1, col=i+1)

fig.update_layout(
    title=dict(text='<b>Resíduo do Consumo de Energia por Região — Brasil</b>',
               font=dict(size=15)),
    template='plotly_white',
    height=430, width=1200,
    showlegend=False,
    margin=dict(t=80, b=40))

fig.update_yaxes(title_text='Fator residual', row=1, col=1,
                 gridcolor='rgba(200,200,200,0.3)')
fig.update_xaxes(gridcolor='rgba(200,200,200,0.3)')

fig.show()

![](midia/residuo_regioes.png)

### Observações

O resíduo por região revela que os choques não afetam o Brasil de forma uniforme.

O **COVID (mai/2020)** é o único evento sincronizado em todas as regiões, mas
Sudeste e Sul caem mais fundo pelo peso industrial e comercial.

A **crise de 2008** aparece concentrada no Sul e Sudeste, confirmando que foi
um choque de produção industrial, quase imperceptível no Norte e Nordeste.

O pico de **fevereiro de 2014** aparece simultaneamente no Sudeste e no Sul,
confirmando a onda de calor identificada na decomposição total.

O **Norte** registra o maior pico positivo da série em fev/2020, coincidindo
com a retomada da indústria de metais não ferrosos após o embargo de 2018.

O **Nordeste** tem pico isolado em dez/2019, provavelmente combinação de calor
acima da média, fim de ano comercial e variações industriais pontuais.

O **Centro-Oeste** é o mais ruidoso dos cinco, reflexo da alta variância natural
de uma economia agroindustrial.

## 8. Conclusão

Vinte anos de dados revelam que o Brasil não tem um padrão único de consumo
de energia :tem vários, sobrepostos e frequentemente opostos.

**O consumo cresceu de forma desigual entre segmentos e regiões.** O residencial
avançou de forma contínua e resiliente, impulsionado pela universalização do
acesso. O industrial foi o mais volátil, concentrando as quedas mais severas
nas crises de 2008 e 2014. O Centro-Oeste dobrou seu consumo em duas décadas,
enquanto o Sudeste, já maduro, cresceu apenas 40%.

**O industrial é o segmento mais sensível a crises econômicas.** As recessões
de 2008 e 2014 aparecem com clareza na tendência e no resíduo industrial, quase
invisíveis no residencial. O comercial, por sua vez, mostrou-se o mais vulnerável
a choques de mobilidade , como ficou evidente no lockdown de 2020.

**O Brasil não tem uma sazonalidade única.** Sul e Sudeste consomem mais no
verão; Norte e Nordeste, no segundo semestre. Agregar essas curvas numa média
nacional esconde mais do que revela, a sazonalidade "brasileira" é, na prática,
a sazonalidade do Sudeste.

**Vários eventos históricos deixaram marca visível no consumo.** A crise
financeira de 2008, a onda de calor de fevereiro de 2014, o embargo ambiental
da Alunorte em 2018 e a pandemia de 2020 aparecem com precisão na  análise,
sem que fosse necessário conhecê-los previamente. Os dados os encontraram.

**O achado mais surpreendente foi o Pará.** Um único embargo ambiental numa
refinaria de alumina derrubou o consumo de energia de um estado inteiro por
mais de um ano , e essa queda foi grande o suficiente para puxar toda a região
Norte para baixo na análise de tendencia.

## Limitações e Trabalhos Futuros

### Limitações

A principal limitação é a **sazonalidade fixa** da decomposição STL: o modelo
assume que o padrão sazonal se repete identicamente a cada ano. Como o consumo
responde a temperaturas, e as temperaturas variam, parte do que foi interpretado
como evento atípico no resíduo pode ser variação climática interanual não modelada.

A granularidade mensal também limita a análise: eventos de curta duração aparecem
diluídos ou invisíveis na série.

### Trabalhos Futuros

**Modelagem preditiva com SARIMAX** para projetar o consumo dos próximos meses,
incorporando variáveis exógenas como temperatura e crescimento do PIB.

**Correlação entre temperatura e consumo**, cruzando os dados com séries
meteorológicas por estado para quantificar o impacto de cada grau Celsius
adicional, relevante num contexto de mudanças climáticas.

**Análise de carga horária com dados do ONS**, aplicando decomposição de múltipla
sazonalidade (diária, semanal e mensal) para revelar uma camada de complexidade
que a granularidade mensal não captura